# Explainability with SHAP
This notebook demonstrates how SHAP (SHapley Additive exPlanations) is used to interpret model predictions globally and locally.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../../'))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image

from ml.registry.model_registry import ModelRegistry
from ml.explainability.shap_explainer import SHAPExplainer
from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
# Load Pre-Trained Model from Registry
model, preprocessor, fe, meta = ModelRegistry.load_latest_model()
print(f"Loaded {meta['model_type']} version: {meta['version']}")

In [ ]:
# To demonstrate SHAP, we need a sample of processed data
from ml.data.joiner import DataJoiner
df = DataJoiner().create_master_dataset().sample(500, random_state=42)
df_features = fe.transform(df)
X = df_features.drop(columns=['risk_label', 'transaction_id'], errors='ignore')
X_processed = preprocessor.transform(X)

In [ ]:
# 1. Initialize Explainer
explainer = SHAPExplainer(model, X_processed)

In [ ]:
# 2. Global Explanations (Feature Importance)
bar_path, dot_path = explainer.generate_global_explanations(X_processed.sample(100, random_state=42))
Image(filename=dot_path)

In [ ]:
# 3. Local Explanations (Single Transaction)
# Take a highly risky transaction as an example
preds = model.predict(X_processed)
risky_indices = [i for i, p in enumerate(preds) if p == 1]

if risky_indices:
    sample_idx = risky_indices[0]
    single_instance = X_processed.iloc[[sample_idx]]
    
    explanation = explainer.explain_local_instance(single_instance)
    print("\nLocal Explanation for High Risk Transaction:")
    print(f"Base Expected Value: {explanation['base_value']:.2f}\n")
    for factor in explanation['top_factors']:
        print(f"- {factor['feature']}: {factor['impact']} (Contribution: {factor['contribution']:.4f}, Value: {factor['value']:.2f})")
else:
    print("No risky transactions found in sample.")